# 🎨 Zero-Trust Ephemeral Ciphertext Mailbox Worker (`v0.2.0`)

Execute batch image/video inference (`SDXL`, `FLUX.1-schnell`, `Wan 2.1/2.2`) on remote cloud GPUs without leaving plaintext prompts, reference images, or outputs on shared cloud storage:
1. Run **Cell 1** once to install dependencies, disable IPython history persistence, and unlock your passphrase in memory.
2. Paste `KISSSHOT_TASK_HEX` generated by your local desktop console into **Cell 2**.
3. Pull and decrypt the double-AEAD (`AES-256-GCM` outer envelope + `WZ_AES` 256-bit `ZIP_STORED` inner archive) locally — burn-after-reading automatically deletes the remote ciphertext shard.

In [ ]:
# === Cell 1: Dependencies, History Scrubbing & Passphrase Initialization ===
import subprocess, sys, os, io, gc, json, re, hashlib, shutil, sqlite3, time, secrets, getpass

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["DIFFUSERS_VERBOSITY"] = "error"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

def scrub_ipython_history() -> None:
    try:
        ip = get_ipython()  # noqa: F821
        if ip and getattr(ip, "history_manager", None):
            hm = ip.history_manager
            hm.enabled = False
            if getattr(hm, "db", None):
                try:
                    hm.db.execute("DELETE FROM history")
                    hm.db.commit()
                except Exception:
                    pass
        if ip and hasattr(ip, "user_ns"):
            for k in ("In", "_ih"):
                arr = ip.user_ns.get(k)
                if isinstance(arr, list):
                    for idx in range(len(arr)):
                        if "KISSSHOT_TASK_HEX" in str(arr[idx]):
                            arr[idx] = "# [cleared]"
    except Exception:
        pass
    hist_db = os.path.expanduser("~/.ipython/profile_default/history.sqlite")
    if os.path.exists(hist_db):
        try:
            con = sqlite3.connect(hist_db)
            con.execute("DELETE FROM history")
            con.commit()
            con.execute("VACUUM")
            con.close()
        except Exception:
            pass

scrub_ipython_history()

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "diffusers>=0.33.0", "transformers", "accelerate", "sentencepiece",
    "imageio[ffmpeg]", "cryptography", "pyzipper", "huggingface_hub", "hf_transfer"
])

import torch, pyzipper
from PIL import Image
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC

KDF_ITERATIONS = 210_000
SALT_BYTES, NONCE_BYTES, TAG_BYTES = 16, 12, 16
AAD_PAYLOAD = b"kissshot-payload-v2"
AAD_BLOB = b"kissshot-blob-v2"
AAD_ARCHIVE = b"kissshot-archive-v2"
MAX_TTL_SECONDS = 86_400
_SAFE_STAGING_RE = re.compile(r"^_staging/ref_[0-9a-f]{12}\.bin$")
_PIPE_CACHE = {"sig": None, "pipe": None}

def derive_key(pwd: str, salt: bytes) -> bytes:
    kdf = PBKDF2HMAC(algorithm=hashes.SHA256(), length=32, salt=salt, iterations=KDF_ITERATIONS)
    return kdf.derive(pwd.encode("utf-8"))

def decrypt_payload_hex(hex_str: str, pwd: str, aad: bytes = AAD_PAYLOAD) -> dict:
    raw = bytes.fromhex("".join(hex_str.split()))
    if len(raw) <= SALT_BYTES + NONCE_BYTES + TAG_BYTES:
        raise ValueError("Truncated ciphertext")
    pt = AESGCM(derive_key(pwd, raw[:SALT_BYTES])).decrypt(
        raw[SALT_BYTES:SALT_BYTES + NONCE_BYTES],
        raw[SALT_BYTES + NONCE_BYTES:],
        aad,
    )
    parsed = json.loads(pt.decode("utf-8"))
    if isinstance(parsed, dict) and parsed.get("_ks_v") == 2 and "d" in parsed:
        iat = int(parsed.get("_ks_iat", 0) or 0)
        if not iat or abs(time.time() - iat) > MAX_TTL_SECONDS:
            raise ValueError("Expired task payload")
        return parsed["d"]
    return parsed

def encrypt_bytes_aes(raw: bytes, pwd: str, aad: bytes = AAD_BLOB) -> bytes:
    salt, nonce = secrets.token_bytes(SALT_BYTES), secrets.token_bytes(NONCE_BYTES)
    ct = AESGCM(derive_key(pwd, salt)).encrypt(nonce, raw, aad)
    return salt + nonce + ct

def decrypt_bytes_aes(enc: bytes, pwd: str, aad: bytes = AAD_BLOB) -> bytes:
    if len(enc) <= SALT_BYTES + NONCE_BYTES + TAG_BYTES:
        raise ValueError("Truncated encrypted blob")
    return AESGCM(derive_key(pwd, enc[:SALT_BYTES])).decrypt(
        enc[SALT_BYTES:SALT_BYTES + NONCE_BYTES],
        enc[SALT_BYTES + NONCE_BYTES:],
        aad,
    )

def pack_encrypted_zip(items: list, pwd: str, meta: dict) -> bytes:
    buf = io.BytesIO()
    pwd_b = pwd.encode("utf-8")
    manifest = {"version": 2, "created_at": int(time.time()), "files": [], "meta": meta}
    with pyzipper.AESZipFile(buf, "w", compression=pyzipper.ZIP_STORED, encryption=pyzipper.WZ_AES) as zf:
        zf.setpassword(pwd_b)
        zf.setencryption(pyzipper.WZ_AES, nbits=256)
        for i, (ext, data) in enumerate(items):
            inner = f"chunk_{i:03d}.dat"
            zf.writestr(inner, data)
            manifest["files"].append({"inner": inner, "ext": ext.lstrip(".") or "bin"})
        zf.writestr("manifest.json", json.dumps(manifest, ensure_ascii=False).encode("utf-8"))
    return encrypt_bytes_aes(buf.getvalue(), pwd, aad=AAD_ARCHIVE)

SESSION_PASSPHRASE = getpass.getpass("🔑 Enter your Kiss-Shot passphrase (in-memory only): ").strip()
assert SESSION_PASSPHRASE, "Passphrase cannot be empty."
print("✅ Worker ready on GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# === Cell 2: Scoped Decrypt of KISSSHOT_TASK_HEX, In-Memory Inference & Double-AEAD Relay ===
KISSSHOT_TASK_HEX = ""

def _run_scoped_media_task(raw_hex: str) -> None:
    assert raw_hex.strip(), "Paste KISSSHOT_TASK_HEX from your local console first."
    task = decrypt_payload_hex(raw_hex, SESSION_PASSPHRASE)
    assert task.get("type") == "kissshot_media_infer", "Invalid task payload type."

    target_repo = str(task["src_repo"])
    preset_key = str(task.get("preset_key", "preset"))
    arch = str(task["arch"])
    variant = task.get("variant")
    hf_token = str(task["hf_token"])
    mailbox_repo = str(task["mailbox_repo"])
    prompt_str = str(task["prompt"])
    neg_str = str(task.get("negative_prompt", ""))
    seed = int(task["seed"]) if int(task["seed"]) >= 0 else secrets.randbits(31)
    width, height = (int(task["width"]) // 16) * 16, (int(task["height"]) // 16) * 16
    steps, cfg_val = int(task["steps"]), float(task["cfg"])
    batch_cnt = max(1, int(task.get("batch", 1)))
    frames_req = int(task.get("frames", 81))
    ref_staging = str(task.get("ref_staging_blob", ""))
    del task

    api = HfApi(token=hf_token)
    repo_sig = hashlib.sha256(f"{target_repo}|{variant}".encode("utf-8")).hexdigest()
    ckpt_dir = f"/tmp/kissshot_ckpt/{repo_sig[:12]}"

    try:
        if _PIPE_CACHE["sig"] != repo_sig:
            if _PIPE_CACHE["pipe"] is not None:
                del _PIPE_CACHE["pipe"]
                _PIPE_CACHE["pipe"] = None
                _PIPE_CACHE["sig"] = None
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            if not os.path.isfile(os.path.join(ckpt_dir, ".ready")):
                os.makedirs(ckpt_dir, exist_ok=True)
                try:
                    snapshot_download(repo_id=target_repo, local_dir=ckpt_dir, ignore_patterns=["*.md", "README*", ".git*"], token=False)
                except Exception:
                    snapshot_download(repo_id=target_repo, local_dir=ckpt_dir, ignore_patterns=["*.md", "README*", ".git*"], token=hf_token)
                shutil.rmtree(os.path.join(ckpt_dir, ".cache"), ignore_errors=True)
                open(os.path.join(ckpt_dir, ".ready"), "w").write(repo_sig)

            if arch == "sdxl":
                from diffusers import StableDiffusionXLPipeline
                kw = {"torch_dtype": torch.float16, "use_safetensors": True, "local_files_only": True}
                if variant:
                    kw["variant"] = variant
                pipe = StableDiffusionXLPipeline.from_pretrained(ckpt_dir, **kw).to("cuda")
            elif arch == "flux":
                from diffusers import FluxPipeline
                pipe = FluxPipeline.from_pretrained(ckpt_dir, torch_dtype=torch.bfloat16, local_files_only=True)
                pipe.enable_model_cpu_offload()
            elif arch in ("wan_t2v", "wan_ti2v"):
                from diffusers import WanPipeline, AutoencoderKLWan
                vae = AutoencoderKLWan.from_pretrained(ckpt_dir, subfolder="vae", torch_dtype=torch.float32, local_files_only=True)
                pipe = WanPipeline.from_pretrained(ckpt_dir, vae=vae, torch_dtype=torch.bfloat16, local_files_only=True)
                pipe.enable_model_cpu_offload()
            else:
                raise ValueError(f"Unsupported architecture: {arch}")
            _PIPE_CACHE["sig"] = repo_sig
            _PIPE_CACHE["pipe"] = pipe

        pipe = _PIPE_CACHE["pipe"]
        out_blobs = []

        if ref_staging:
            if not _SAFE_STAGING_RE.fullmatch(ref_staging):
                raise ValueError("Unsafe staging blob path rejected")
            import tempfile
            with tempfile.TemporaryDirectory(dir="/dev/shm" if os.path.isdir("/dev/shm") else None) as td:
                enc_p = hf_hub_download(repo_id=mailbox_repo, filename=ref_staging, repo_type="model", token=hf_token, local_dir=td)
                _ = decrypt_bytes_aes(open(enc_p, "rb").read(), SESSION_PASSPHRASE, aad=AAD_BLOB)
            try:
                api.delete_file(path_in_repo=ref_staging, repo_id=mailbox_repo, repo_type="model", commit_message="cleanup staging")
            except Exception:
                pass

        if arch in ("sdxl", "flux"):
            for idx in range(batch_cnt):
                gen = torch.Generator(device="cpu").manual_seed(seed + idx)
                kw = dict(prompt=prompt_str, width=width, height=height, num_inference_steps=steps, guidance_scale=cfg_val, generator=gen)
                if arch == "sdxl":
                    kw["negative_prompt"] = neg_str
                img = pipe(**kw).images[0]
                mem = io.BytesIO()
                img.save(mem, format="PNG")
                out_blobs.append(("png", mem.getvalue()))
        else:
            import tempfile
            from diffusers.utils import export_to_video
            frames_cnt = ((max(17, frames_req) - 1) // 4) * 4 + 1
            gen = torch.Generator(device="cpu").manual_seed(seed)
            frames = pipe(
                prompt=prompt_str,
                negative_prompt=neg_str,
                width=width,
                height=height,
                num_frames=frames_cnt,
                guidance_scale=cfg_val,
                num_inference_steps=steps,
                generator=gen,
            ).frames[0]
            shm = "/dev/shm" if os.path.isdir("/dev/shm") else tempfile.gettempdir()
            tmp_mp4 = os.path.join(shm, f"ks_{secrets.token_hex(6)}.mp4")
            try:
                export_to_video(frames, tmp_mp4, fps=16)
                with open(tmp_mp4, "rb") as vf:
                    out_blobs.append(("mp4", vf.read()))
            finally:
                if os.path.exists(tmp_mp4):
                    os.remove(tmp_mp4)

        enc_zip = pack_encrypted_zip(out_blobs, SESSION_PASSPHRASE, meta={"seed": seed, "preset": preset_key})
        del out_blobs
        remote_blob = f"_inbox/pkt_{int(time.time())}_{secrets.token_hex(4)}.bin"
        api.create_repo(repo_id=mailbox_repo, repo_type="model", private=True, exist_ok=True)
        api.upload_file(path_or_fileobj=enc_zip, path_in_repo=remote_blob, repo_id=mailbox_repo, repo_type="model", commit_message="sync artifact")
        print(f"✅ Done (seed={seed}). Double-AEAD archive uploaded to {mailbox_repo}/{remote_blob}.")
    finally:
        prompt_str = ""
        neg_str = ""
        hf_token = ""
        del api

_tmp_ks_hex = KISSSHOT_TASK_HEX
KISSSHOT_TASK_HEX = ""
scrub_ipython_history()
try:
    _run_scoped_media_task(_tmp_ks_hex)
finally:
    _tmp_ks_hex = ""
    scrub_ipython_history()
    gc.collect()